In [58]:
# 1. Environment Setup
!pip install --no-cache-dir -U transformers datasets peft trl accelerate bitsandbytes

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [59]:
# 2. Dataset Loading & Inspection

import json
import pandas as pd
from pathlib import Path

# Path to dataset
# dataset_path = Path('../data/dataset.jsonl')
dataset_path = Path('/kaggle/input/major-jsonl/dataset.jsonl')

# Read JSONL dataset (skip non-JSON lines)
samples = []
with open(dataset_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line or not line.startswith('{'):
            continue  # skip comments, blank lines, or metadata
        try:
            samples.append(json.loads(line))
        except json.JSONDecodeError as e:
            print("Skipping line due to JSON error:", e)

# Convert to DataFrame for inspection
sample_df = pd.DataFrame(samples)
print('Columns:', sample_df.columns.tolist())
print('Sample rows:')
display(sample_df.head())

# Identify input/output columns
input_col = None
output_col = None
for col in sample_df.columns:
    if 'prompt' in col or 'desc' in col:
        input_col = col
    if 'code' in col:
        output_col = col
print(f"Input column: {input_col}")
print(f"Output column: {output_col}")

Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due to JSON error: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Skipping line due t

,prompt,code
0,show the integral of y = 1/x shaded from x=1 t...,from manim import *\nimport numpy as np\n\ncla...
1,"show signed area under y = x on [-1, 1].",from manim import *\n\nclass SignedAreaLinear(...
2,"compare exact area under y = x^2 on [0,1] with...",from manim import *\n\nclass CompareExactEstim...
3,show midpoint Riemann sum approximation for y ...,from manim import *\nimport numpy as np\n\ncla...
4,Visualize and compute the area between y = x a...,from manim import *\nimport numpy as np\n\ncla...


Input column: prompt
Output column: code


In [60]:
# 3. Preprocessing: Instruction-Response Formatting

from datasets import Dataset

# Use the columns detected above
def format_instruction(example):
    # SFTTrainer expects a dict with 'prompt' and 'completion' keys by default
    return {
        'prompt': example[input_col].strip(),
        'completion': example[output_col].strip(),
        'text': f"<s>[INST] {example[input_col].strip()} [/INST]\n{example[output_col].strip()}</s>"
    }

# Apply formatting
dataset = Dataset.from_pandas(sample_df)
dataset = dataset.map(format_instruction)

# Show a sample
print('Formatted sample:')
print(dataset[0])

Map:   0%|          | 0/367 [00:00<?, ? examples/s]

Formatted sample:
{'prompt': 'show the integral of y = 1/x shaded from x=1 to x=e.', 'code': 'from manim import *\nimport numpy as np\n\nclass IntegralOneOverX(Scene):\n    def construct(self):\n        axes = Axes(\n            x_range=[0.5, np.e + 1, 0.5],\n            y_range=[0, 1.5, 0.5],\n            x_length=8,\n            y_length=5,\n            axis_config={"include_numbers": True}\n        ).to_edge(DOWN)\n\n        curve = axes.plot(lambda x: 1 / x, color=BLUE)\n        label = MathTex(r"y = \\frac{1}{x}").next_to(curve, UP + RIGHT)\n\n        # Shade area under curve from x=1 to x=e\n        area = axes.get_area(curve, x_range=[1, np.e], color=GREEN, opacity=0.5)\n\n        self.play(Create(axes), Create(curve), Write(label))\n        self.wait(1)\n        self.play(FadeIn(area))\n        self.wait(3)', 'completion': 'from manim import *\nimport numpy as np\n\nclass IntegralOneOverX(Scene):\n    def construct(self):\n        axes = Axes(\n            x_range=[0.5, np.e + 

In [62]:
# 4. Model & Tokenizer Loading (Quantized)

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization config (bitsandbytes)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

# Load tokenizer
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# Load model (quantized)
print('Loading model (4-bit quantized)...')
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Enable gradient checkpointing for memory savings
model.gradient_checkpointing_enable()

# Enable flash attention if available
try:
    model.enable_flash_attn()
    print('Flash attention enabled!')
except Exception:
    print('Flash attention not available.')

Loading tokenizer...
Loading model (4-bit quantized)...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Flash attention not available.


In [64]:
# 5. LoRA (QLoRA) & Trainer Setup

from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
from transformers import TrainingArguments
import os

# LoRA config (QLoRA, memory efficient)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# Output dir for LoRA adapters
output_dir = "../outputs/mistral/lora"
os.makedirs(output_dir, exist_ok=True)

# Training arguments 
training_args = TrainingArguments(
    per_device_train_batch_size=1,  
    gradient_accumulation_steps=8,  # Effective batch size = 8
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    output_dir=output_dir,
    save_strategy="epoch",
    report_to="none",
    save_total_limit=1,
    max_steps=-1,  # Use -1 instead of None to avoid TypeError
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
)

# SFTTrainer for instruction tuning
# Use the default columns: 'prompt' and 'completion' (created in preprocessing)
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=training_args,
)

Adding EOS to train dataset:   0%|          | 0/367 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/367 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/367 [00:00<?, ? examples/s]

In [ ]:
# 6. Training Loop

# Clear CUDA cache before training to avoid OOM
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Train
trainer.train()

# Save LoRA adapter
trainer.model.save_pretrained(output_dir)
print(f"LoRA adapter saved to {output_dir}")

Step,Training Loss
10,0.461300
20,0.322000
30,0.294900
40,0.276000
50,0.262200
60,0.183800
70,0.200600
80,0.213800


In [ ]:
# 7. Inference Test: Generate Manim Code

def generate_manim_code(prompt, model, tokenizer, max_new_tokens=512):
    model.eval()
    input_text = f"<s>[INST] {prompt.strip()} [/INST]"
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            eos_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the response (after [/INST])
    if '[/INST]' in decoded:
        decoded = decoded.split('[/INST]', 1)[-1].strip()
    return decoded

# Example prompt
test_prompt = "Create a Manim animation explaining Pythagoras theorem"
response = generate_manim_code(test_prompt, trainer.model, tokenizer)
print("Prompt:\n", test_prompt)
print("\nGenerated Manim code:\n", response)

# Save output for inspection
with open("../outputs/mistral/test_output.py", "w", encoding="utf-8") as f:
    f.write(response)
print("\nSample output saved to ../outputs/mistral/test_output.py")

In [ ]:
# 8. Early Stopping, Metrics, and Training/Validation Curves

from transformers import EarlyStoppingCallback, TrainerCallback
from datasets import load_metric
import matplotlib.pyplot as plt
import numpy as np
import os

# Add validation split (10% of data)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
val_dataset = dataset['test']

# Define compute_metrics function for ROUGE, BLEU, accuracy
rouge = load_metric('rouge')
bleu = load_metric('bleu')

# For accuracy: compare exact string match
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # ROUGE
    rouge_result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    # BLEU
    bleu_result = bleu.compute(predictions=[[pred.split()] for pred in decoded_preds], references=[ [label.split()] for label in decoded_labels])
    # Accuracy (exact match)
    acc = accuracy_score(decoded_labels, decoded_preds)
    return {
        'rougeL': rouge_result['rougeL'].mid.fmeasure,
        'bleu': bleu_result['bleu'],
        'accuracy': acc
    }

# Early stopping callback
early_stopping = EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.0)

# Custom callback to log metrics for plotting
class MetricsLoggerCallback(TrainerCallback):
    def __init__(self):
        self.train_loss = []
        self.eval_loss = []
        self.eval_rouge = []
        self.eval_bleu = []
        self.eval_acc = []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            if 'loss' in logs:
                self.train_loss.append(logs['loss'])
            if 'eval_loss' in logs:
                self.eval_loss.append(logs['eval_loss'])
            if 'eval_rougeL' in logs:
                self.eval_rouge.append(logs['eval_rougeL'])
            if 'eval_bleu' in logs:
                self.eval_bleu.append(logs['eval_bleu'])
            if 'eval_accuracy' in logs:
                self.eval_acc.append(logs['eval_accuracy'])

metrics_logger = MetricsLoggerCallback()

# Re-create trainer with validation, metrics, and callbacks
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=lora_config,
    args=training_args,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping, metrics_logger],
)

# Train
trainer.train()

# Save LoRA adapter
trainer.model.save_pretrained(output_dir)
print(f"LoRA adapter saved to {output_dir}")

# Plot and save metrics
os.makedirs('../outputs/mistral/metrics', exist_ok=True)
plt.figure(figsize=(8,5))
plt.plot(metrics_logger.train_loss, label='Train Loss')
plt.plot(metrics_logger.eval_loss, label='Validation Loss')
plt.xlabel('Logging Step')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.savefig('../outputs/mistral/metrics/loss_curve.png')
plt.close()

plt.figure(figsize=(8,5))
plt.plot(metrics_logger.eval_rouge, label='ROUGE-L')
plt.xlabel('Eval Step')
plt.ylabel('ROUGE-L')
plt.title('Validation ROUGE-L Score')
plt.legend()
plt.savefig('../outputs/mistral/metrics/rouge_curve.png')
plt.close()

plt.figure(figsize=(8,5))
plt.plot(metrics_logger.eval_bleu, label='BLEU')
plt.xlabel('Eval Step')
plt.ylabel('BLEU')
plt.title('Validation BLEU Score')
plt.legend()
plt.savefig('../outputs/mistral/metrics/bleu_curve.png')
plt.close()

plt.figure(figsize=(8,5))
plt.plot(metrics_logger.eval_acc, label='Accuracy')
plt.xlabel('Eval Step')
plt.ylabel('Accuracy')
plt.title('Validation Accuracy')
plt.legend()
plt.savefig('../outputs/mistral/metrics/accuracy_curve.png')
plt.close()

print('Saved all metric plots to ../outputs/mistral/metrics/')
